In [2]:
import os
import sys

# Detect if executing inside Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/Arkadyg27/TimeSeriesProject.git"
    PROJECT_DIR = "/content/TimeSeriesProject"

    os.chdir('/content')
    if not os.path.exists(PROJECT_DIR):
        !git clone {REPO_URL}
    else:
        os.chdir(PROJECT_DIR)
        !git pull

    os.chdir(PROJECT_DIR)
    print("Google Colab detected. Working directory set to:", os.getcwd())
else:
    print("Running locally. Working directory set to:", os.getcwd())


Cloning into 'TimeSeriesProject'...
remote: Enumerating objects: 250, done.
remote: Counting objects: 100% (159/159), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 250 (delta 90), reused 131 (delta 65), pack-reused 91 (from 1)
Receiving objects: 100% (250/250), 65.67 MiB | 18.74 MiB/s, done.
Resolving deltas: 100% (109/109), done.
Google Colab detected. Working directory set to: /content/TimeSeriesProject


In [3]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install mlflow rasterio pymannkendall
else:
    print("Local environment detected. Make sure dependencies are installed.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 113.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 114.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.2/123.2 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
import sys
IN_COLAB = 'google.colab' in sys.modules

# 1. Native Colab Authentication
if IN_COLAB:
    try:
        from google.colab import auth
        auth.authenticate_user()
    except Exception as e:
        print('Colab auth notice:', e)

import ee

# Team Earth Engine Projects
PROJECT_IDS = ['889258893131', 'timeseriesproject-503021']

initialized = False
for proj in PROJECT_IDS:
    try:
        ee.Initialize(project=proj)
        print(f'Earth Engine initialized successfully using project: "{proj}"')
        initialized = True
        break
    except Exception:
        continue

if not initialized:
    try:
        ee.Initialize()
        print('Earth Engine initialized using account default project!')
    except Exception:
        print('Prompting interactive Earth Engine authentication...')
        ee.Authenticate()
        ee.Initialize()


Earth Engine initialized successfully using project: "timeseriesproject-503021"


In [5]:
import os, sys
IN_COLAB = 'google.colab' in sys.modules

# Sync full and partial download checkpoints from Google Drive to preserve download progress
if IN_COLAB:
    from google.colab import drive
    try:
        drive.mount('/content/drive', force_remount=False)
    except ValueError as e:
        if 'Mountpoint must not already contain files' in str(e):
            print('Detected dirty mountpoint. Cleaning up...')
            import shutil
            shutil.rmtree('/content/drive', ignore_errors=True)
            drive.mount('/content/drive', force_remount=True)
        else:
            raise
    drive_cache_dir = '/content/drive/MyDrive/Study/MSc_CE_BGU/Time Series Analysis/Final Project/TimeSeriesProject'

    if os.path.exists(drive_cache_dir):
        import glob, shutil
        for f in glob.glob(f"{drive_cache_dir}/*.parquet"):
            dest = f"/content/TimeSeriesProject/{os.path.basename(f)}"
            if not os.path.exists(dest) or os.path.getsize(dest) < 100000:
                shutil.copy2(f, dest)
        print('Smart-Synced dataset caches from Google Drive (preventing corrupt 0-byte files)!')
    else:
        !mkdir -p "/content/drive/MyDrive/Study/MSc_CE_BGU/Time Series Analysis/Final Project/TimeSeriesProject"


Mounted at /content/drive
Smart-Synced dataset caches from Google Drive (preventing corrupt 0-byte files)!


In [6]:
%env MLFLOW_ALLOW_FILE_STORE=true
import os, sys
import mlflow
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    try:
        drive.mount('/content/drive', force_remount=False)
    except ValueError as e:
        if 'Mountpoint must not already contain files' in str(e):
            print('Detected dirty mountpoint. Cleaning up...')
            import shutil
            shutil.rmtree('/content/drive', ignore_errors=True)
            drive.mount('/content/drive', force_remount=True)
        else:
            raise

    # Define project path in Google Drive
    DRIVE_PROJECT_PATH = '/content/drive/MyDrive/Study/MSc_CE_BGU/Time Series Analysis/Final Project/TimeSeriesProject'
    MLRUNS_DIR = os.path.join(DRIVE_PROJECT_PATH, 'mlruns')
    os.makedirs(MLRUNS_DIR, exist_ok=True)

    # Direct MLflow to log persistently to Google Drive
    os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
    # Forcing MLflow to use the file store exclusively to avoid SQLite cross-platform corruption
    os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
    if "MLFLOW_TRACKING_URI" in os.environ:
        del os.environ["MLFLOW_TRACKING_URI"]
    mlflow.set_tracking_uri(f"file:///{MLRUNS_DIR}")

    import IPython
    IPython.get_ipython().run_line_magic('env', f'MLFLOW_TRACKING_URI=file:///{MLRUNS_DIR}')
    print(f"MLflow tracking initialized! Runs will be saved to: {MLRUNS_DIR}")
else:
    print("Local environment: MLflow tracking locally.")


env: MLFLOW_ALLOW_FILE_STORE=true
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
env: MLFLOW_TRACKING_URI=file:////content/drive/MyDrive/Study/MSc_CE_BGU/Time Series Analysis/Final Project/TimeSeriesProject/mlruns
MLflow tracking initialized! Runs will be saved to: /content/drive/MyDrive/Study/MSc_CE_BGU/Time Series Analysis/Final Project/TimeSeriesProject/mlruns


In [7]:
# ========================================================
# SMART PREPROCESSING: Skip if files already exist in Drive
# ========================================================
import os, shutil, sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    DRIVE_PREPROCESS = '/content/drive/MyDrive/Study/MSc_CE_BGU/Time Series Analysis/Final Project/TimeSeriesProject/Preprocess'
    LOCAL_PREPROCESS = 'Preprocess'

    # Check if preprocessed files already exist in Google Drive
    check_file = os.path.join(DRIVE_PREPROCESS, 'Altamira_NDVI_CenteredMatrix.parquet')

    if os.path.exists(check_file):
        print("Found preprocessed data in Google Drive! Skipping preprocessing...")
        # Instantly link Google Drive files to local workspace
        os.makedirs(LOCAL_PREPROCESS, exist_ok=True)
        for file_name in os.listdir(DRIVE_PREPROCESS):
            src = os.path.join(DRIVE_PREPROCESS, file_name)
            dst = os.path.join(LOCAL_PREPROCESS, file_name)
            if not os.path.exists(dst):
                os.symlink(src, dst)
        DRIVE_TIFF = os.path.join(DRIVE_PROJECT_PATH, "Tiff")
        LOCAL_TIFF = "Tiff"
        if os.path.exists(DRIVE_TIFF):
            import shutil
            if os.path.exists(LOCAL_TIFF) and not os.path.islink(LOCAL_TIFF):
                shutil.rmtree(LOCAL_TIFF)
            if not os.path.exists(LOCAL_TIFF):
                os.symlink(DRIVE_TIFF, LOCAL_TIFF)
        print("All preprocessed data linked! Ready to train.")
    else:
        print("First-time run: Preprocessing data...")
        !python run_preprocessing.py

        # Save a backup to Google Drive so you never run it again
        shutil.copytree(LOCAL_PREPROCESS, DRIVE_PREPROCESS, dirs_exist_ok=True)
        print("Preprocessed files backed up to Google Drive for future sessions!")
else:
    print("Local environment: Running preprocessing...")
    !python run_preprocessing.py


Found preprocessed data in Google Drive! Skipping preprocessing...
All preprocessed data linked! Ready to train.


In [8]:
!python run_baseline_all.py


Starting Baseline Experiments for all datasets...

Running Baseline for Altamira (NDVI) with alpha=1.0

--- Running Z-Score Baseline (leak_free=False) ---
Skipping: Baseline already run! Found cached result at Tiff/leaky/Baseline/Altamira_NDVI_Altamira_NDVI_Baseline_leaky.tif

--- Running Z-Score Baseline (leak_free=True) ---
Skipping: Baseline already run! Found cached result at Tiff/leak_free/Baseline/Altamira_NDVI_Altamira_NDVI_Baseline_leakfree.tif

Running Baseline for Brumadinho (NDVI) with alpha=1.0

--- Running Z-Score Baseline (leak_free=False) ---
Skipping: Baseline already run! Found cached result at Tiff/leaky/Baseline/Brumadinho_NDVI_Brumadinho_NDVI_Baseline_leaky.tif

--- Running Z-Score Baseline (leak_free=True) ---
Skipping: Baseline already run! Found cached result at Tiff/leak_free/Baseline/Brumadinho_NDVI_Brumadinho_NDVI_Baseline_leakfree.tif

Running Baseline for Brumadinho (NDWI) with alpha=1.0

--- Running Z-Score Baseline (leak_free=False) ---
Skipping: Baseline 

In [9]:
!python Altamira_Modis_repro.py

Starting experiments for Altamira (NDVI)...
Loading cached dataset from data_Altamira_ndvi.parquet...
Raw data loaded. Shape: (64449, 276)

--- Running Isolation Forest (leak_free=False) ---
Skipping: Model already trained! Found cached result at Tiff/leaky/IsolationForest/Altamira_NDVI_IsolationForest_leaky_est_20.tif
Skipping: Model already trained! Found cached result at Tiff/leaky/IsolationForest/Altamira_NDVI_IsolationForest_leaky_est_40.tif
Skipping: Model already trained! Found cached result at Tiff/leaky/IsolationForest/Altamira_NDVI_IsolationForest_leaky_est_60.tif
Skipping: Model already trained! Found cached result at Tiff/leaky/IsolationForest/Altamira_NDVI_IsolationForest_leaky_est_80.tif

--- Running Isolation Forest (leak_free=True) ---
Skipping: Model already trained! Found cached result at Tiff/leak_free/IsolationForest/Altamira_NDVI_IsolationForest_leakfree_est_20.tif
Skipping: Model already trained! Found cached result at Tiff/leak_free/IsolationForest/Altamira_NDVI_

In [10]:
!python Brumadinho_Sentinel_repro.py

Starting experiments for Brumadinho...

==================== Band: NDWI ====================
Cache file data_Brumadinho_ndwi.parquet not found. Querying Earth Engine (this may take a few minutes)...
Found 158 images.
Successfully cached data to data_Brumadinho_ndwi.parquet!
Raw data loaded. Shape: (99540, 157)

--- Running Isolation Forest (leak_free=False) ---
Skipping: Model already trained! Found cached result at Tiff/leaky/IsolationForest/Brumadinho_NDWI_IsolationForest_leaky_est_20.tif
Skipping: Model already trained! Found cached result at Tiff/leaky/IsolationForest/Brumadinho_NDWI_IsolationForest_leaky_est_40.tif
Skipping: Model already trained! Found cached result at Tiff/leaky/IsolationForest/Brumadinho_NDWI_IsolationForest_leaky_est_60.tif
Skipping: Model already trained! Found cached result at Tiff/leaky/IsolationForest/Brumadinho_NDWI_IsolationForest_leaky_est_80.tif
Skipping: Model already trained! Found cached result at Tiff/leaky/IsolationForest/Brumadinho_NDWI_Isolation

In [11]:
!python Mariana_Landsat_repro.py

Starting experiments for Mariana...

==================== Band: GVMI ====================
Loading cached dataset from data_Mariana_gvmi.parquet...
Raw data loaded. Shape: (52863, 122)

--- Running Isolation Forest (leak_free=False) ---
Skipping: Model already trained! Found cached result at Tiff/leaky/IsolationForest/Mariana_GVMI_IsolationForest_leaky_est_20.tif
Skipping: Model already trained! Found cached result at Tiff/leaky/IsolationForest/Mariana_GVMI_IsolationForest_leaky_est_40.tif
Skipping: Model already trained! Found cached result at Tiff/leaky/IsolationForest/Mariana_GVMI_IsolationForest_leaky_est_60.tif
Skipping: Model already trained! Found cached result at Tiff/leaky/IsolationForest/Mariana_GVMI_IsolationForest_leaky_est_80.tif
Skipping: Model already trained! Found cached result at Tiff/leaky/IsolationForest/Mariana_GVMI_IsolationForest_leaky_est_100.tif

--- Running Isolation Forest (leak_free=True) ---
Skipping: Model already trained! Found cached result at Tiff/leak_f

In [12]:
!python train_deep.py

==================== Deep Learning: Altamira NDVI (LSTM Autoencoder) ====================
Loading precomputed centered matrix...
Extracting Time-Aware Features (Velocity, Acceleration, Rolling Stats)...
Calculating velocity...
Calculating acceleration...
Calculating rolling stats (window=3)...
Stacking features into tensor...
--- USING DEVICE: cuda ---
Training up to 10 epochs...
Epoch [1/10], Loss: 0.031253, Time: 3.17s
Epoch [2/10], Loss: 0.022392, Time: 2.58s
Epoch [3/10], Loss: 0.022389, Time: 2.58s
Epoch [4/10], Loss: 0.022387, Time: 2.71s
Epoch [5/10], Loss: 0.022384, Time: 3.06s
Epoch [6/10], Loss: 0.022379, Time: 2.60s
Epoch [7/10], Loss: 0.022377, Time: 2.61s
Epoch [8/10], Loss: 0.022375, Time: 2.63s
Epoch [9/10], Loss: 0.022378, Time: 2.72s
Epoch [10/10], Loss: 0.022372, Time: 2.87s
Evaluating reconstruction errors at Epoch 10...
Generating MLflow run for Ep10_Pct90...
/usr/local/lib/python3.12/dist-packages/osgeo/osr.py:410: FutureWarning: Neither osr.UseExceptions() nor osr

In [14]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Save full parquet datasets and partial download checkpoints to Google Drive
    !mkdir -p "/content/drive/MyDrive/Study/MSc_CE_BGU/Time Series Analysis/Final Project/TimeSeriesProject"
    !cp -u *.parquet "/content/drive/MyDrive/Study/MSc_CE_BGU/Time Series Analysis/Final Project/TimeSeriesProject/" 2>/dev/null || true
    print('Backed up parquet datasets and checkpoints to Google Drive.')

    # Save output results (Tiff & mlruns)
    !mkdir -p "/content/drive/MyDrive/Study/MSc_CE_BGU/Time Series Analysis/Final Project/TimeSeriesProject"
    !cp -r Tiff/ "/content/drive/MyDrive/Study/MSc_CE_BGU/Time Series Analysis/Final Project/TimeSeriesProject/"
    !cp -r mlruns/ "/content/drive/MyDrive/Study/MSc_CE_BGU/Time Series Analysis/Final Project/TimeSeriesProject/"
    print("Backed up results to /content/drive/MyDrive/Study/MSc_CE_BGU/Time Series Analysis/Final Project/TimeSeriesProject/")
else:
    print("Results and checkpoints are stored locally.")


Backed up parquet datasets and checkpoints to Google Drive.
cp: 'Tiff/' and '/content/drive/MyDrive/Study/MSc_CE_BGU/Time Series Analysis/Final Project/TimeSeriesProject/Tiff' are the same file
cp: cannot stat 'mlruns/': No such file or directory
Backed up results to /content/drive/MyDrive/Study/MSc_CE_BGU/Time Series Analysis/Final Project/TimeSeriesProject/
